In [1]:
from ALLCools.mcds import MCDS

mcds0 = MCDS.open("/ceph/MethDev/pbio/andy/JW/section_4/All_mcds_dmw3_0.mcds")
mcds1 = MCDS.open("/ceph/MethDev/pbio/andy/JW/section_4/All_mcds_dmw3_1.mcds")

print(mcds0)
print("---")
print(mcds1)

<xarray.MCDS>
Dimensions:      (cell: 3036, count_type: 2, dmw: 22467, mc_type: 4)
Coordinates:
  * cell         (cell) <U27 '240614_mct_1_1_P2-4-E5-M20' ... '240614_mct_1_4...
  * count_type   (count_type) <U3 'mc' 'cov'
  * dmw          (dmw) <U9 'dmw=1' 'dmw=2' 'dmw=3' ... 'dmw=22466' 'dmw=22467'
    dmw_chrom    (dmw) int64 1 1 1 1 1 1 1 1 1 1 1 1 ... 5 5 5 5 5 5 5 5 5 5 5 5
    dmw_end      (dmw) int64 25400 27200 30200 ... 26961900 26962400 26968100
    dmw_start    (dmw) int64 23901 25601 27401 ... 26961501 26962101 26967701
  * mc_type      (mc_type) <U3 'CHH' 'CHG' 'CHN' 'CGN'
    strand_type  <U4 'both'
Data variables:
    dmw_da       (cell, dmw, mc_type, count_type) uint32 dask.array<chunksize=(190, 2809, 1, 1), meta=np.ndarray>
Attributes:
    obs_dim:  cell
    var_dim:  null
---
<xarray.MCDS>
Dimensions:      (cell: 3035, count_type: 2, dmw: 22467, mc_type: 4)
Coordinates:
  * cell         (cell) <U27 '240614_mct_1_4_P2-1-I15-F1' ... '240614_mct_1_3...
  * count_type   (

In [2]:
# Compare key dims
for name, m in [("0", mcds0), ("1", mcds1)]:
    print(f"mcds{name}: {m.dims}")
    print(f"  mc_types:    {m.coords['mc_type'].values.tolist()}")
    print(f"  count_types: {m.coords['count_type'].values.tolist()}")
    print(f"  strand:      {m.coords['strand_type'].values.tolist()}")
    print(f"  n_dmw:       {m.dims['dmw']}")
    print(f"  n_cells:     {m.dims['cell']}")

mcds0: Frozen({'cell': 3036, 'count_type': 2, 'dmw': 22467, 'mc_type': 4})
  mc_types:    ['CHH', 'CHG', 'CHN', 'CGN']
  count_types: ['mc', 'cov']
  strand:      both
  n_dmw:       22467
  n_cells:     3036
mcds1: Frozen({'cell': 3035, 'count_type': 2, 'dmw': 22467, 'mc_type': 4})
  mc_types:    ['CHH', 'CHG', 'CHN', 'CGN']
  count_types: ['mc', 'cov']
  strand:      both
  n_dmw:       22467
  n_cells:     3035


In [3]:
# Are the DMW regions identical between files? (they should be, if chunked by cell)
import numpy as np
same_chrom = np.array_equal(mcds0.dmw_chrom.values, mcds1.dmw_chrom.values)
same_start = np.array_equal(mcds0.dmw_start.values, mcds1.dmw_start.values)
print(f"DMW regions identical: chrom={same_chrom}, start={same_start}")

# Cell overlap
c0 = set(mcds0.get_index("cell"))
c1 = set(mcds1.get_index("cell"))
print(f"cells in 0: {len(c0)}, in 1: {len(c1)}, overlap: {len(c0 & c1)}")

DMW regions identical: chrom=True, start=True
cells in 0: 3036, in 1: 3035, overlap: 0


In [5]:
# Concatenate along cell
import xarray as xr
combined = xr.concat([mcds0, mcds1], dim="cell")
print(combined)
print(f"\nTotal cells: {combined.dims['cell']}")  # should be 6071
print(f"Total DMWs:  {combined.dims['dmw']}")

<xarray.MCDS>
Dimensions:      (cell: 6071, count_type: 2, dmw: 22467, mc_type: 4)
Coordinates:
  * cell         (cell) <U27 '240614_mct_1_1_P2-4-E5-M20' ... '240614_mct_1_3...
  * count_type   (count_type) <U3 'mc' 'cov'
  * dmw          (dmw) <U9 'dmw=1' 'dmw=2' 'dmw=3' ... 'dmw=22466' 'dmw=22467'
    dmw_chrom    (dmw) int64 1 1 1 1 1 1 1 1 1 1 1 1 ... 5 5 5 5 5 5 5 5 5 5 5 5
    dmw_end      (dmw) int64 25400 27200 30200 ... 26961900 26962400 26968100
    dmw_start    (dmw) int64 23901 25601 27401 ... 26961501 26962101 26967701
  * mc_type      (mc_type) <U3 'CHH' 'CHG' 'CHN' 'CGN'
    strand_type  <U4 'both'
Data variables:
    dmw_da       (cell, dmw, mc_type, count_type) uint32 dask.array<chunksize=(190, 2809, 1, 1), meta=np.ndarray>
Attributes:
    obs_dim:  null
    var_dim:  null

Total cells: 6071
Total DMWs:  22467


In [6]:
# Quick sanity check: methylation rate distribution for one mc_type
# This loads a small slice into memory — adjust if it's slow
import numpy as np

da = combined['dmw_da']
print("dmw_da shape:", da.shape)
print("dmw_da dims: ", da.dims)
print("\nFirst few cells × first few DMWs (one mc_type):")
# Pick first mc_type, sum across count_type to peek at raw values
print(da.isel(cell=slice(0, 3), dmw=slice(0, 5)).load())

dmw_da shape: (6071, 22467, 4, 2)
dmw_da dims:  ('cell', 'dmw', 'mc_type', 'count_type')

First few cells × first few DMWs (one mc_type):
<xarray.DataArray 'dmw_da' (cell: 3, dmw: 5, mc_type: 4, count_type: 2)>
array([[[[  0,  30],
         [  0,   3],
         [  0,  33],
         [  7,  10]],

        [[  0,  17],
         [  0,   4],
         [  0,  21],
         [  4,   4]],

        [[  0,  54],
         [  0,  10],
         [  0,  64],
         [  8,   8]],

        [[  0,   0],
         [  0,   0],
         [  0,   0],
         [  0,   0]],

...

        [[  0,   0],
         [  0,   0],
         [  0,   0],
         [  0,   0]],

        [[  0,   0],
         [  0,   0],
         [  0,   0],
         [  0,   0]],

        [[  0,   0],
         [  0,   0],
         [  0,   0],
         [  0,   0]],

        [[  0,   0],
         [  0,   0],
         [  0,   0],
         [  0,   0]]]], dtype=uint32)
Coordinates:
  * cell         (cell) <U27 '240614_mct_1_1_P2-4-E5-M20' ... '24061

In [7]:
# Compute methylation rate (mc / cov) for CG context
# Adjust 'CGN' / 'CHN' / 'CHH' depending on what's in mc_type coord
mc_types = combined.coords['mc_type'].values.tolist()
count_types = combined.coords['count_type'].values.tolist()
print(f"mc_types available:    {mc_types}")
print(f"count_types available: {count_types}")

mc_types available:    ['CHH', 'CHG', 'CHN', 'CGN']
count_types available: ['mc', 'cov']


In [8]:
# Example: methylation rate for first mc_type
mc = combined['dmw_da'].sel(mc_type=mc_types[0], count_type='mc')
cov = combined['dmw_da'].sel(mc_type=mc_types[0], count_type='cov')
mrate = (mc / cov.where(cov > 0))  # avoid div by zero
print(mrate)

# Mean coverage per cell across all DMWs
mean_cov_per_cell = cov.mean(dim='dmw').compute()
print("\nMean coverage per cell stats:")
print(f"  median: {float(mean_cov_per_cell.median()):.2f}")
print(f"  mean:   {float(mean_cov_per_cell.mean()):.2f}")

<xarray.DataArray 'dmw_da' (cell: 6071, dmw: 22467)>
dask.array<truediv, shape=(6071, 22467), dtype=float64, chunksize=(190, 2809), chunktype=numpy.ndarray>
Coordinates:
  * cell         (cell) <U27 '240614_mct_1_1_P2-4-E5-M20' ... '240614_mct_1_3...
  * dmw          (dmw) <U9 'dmw=1' 'dmw=2' 'dmw=3' ... 'dmw=22466' 'dmw=22467'
    dmw_chrom    (dmw) int64 1 1 1 1 1 1 1 1 1 1 1 1 ... 5 5 5 5 5 5 5 5 5 5 5 5
    dmw_end      (dmw) int64 25400 27200 30200 ... 26961900 26962400 26968100
    dmw_start    (dmw) int64 23901 25601 27401 ... 26961501 26962101 26967701
    mc_type      <U3 'CHH'
    strand_type  <U4 'both'

Mean coverage per cell stats:
  median: 10.96
  mean:   15.13


['CHH', 'CHG', 'CHN', 'CGN']